# Reproduce the numeric open-model expansion on Colab

Choose **Runtime → Change runtime type → T4 GPU** (or another CUDA GPU), then run the cells in order in a fresh runtime. This notebook runs SmolLM2 1.7B and Granite 3.3 2B on the frozen Breast Cancer and Wine numerical datasets, at zero-shot and four labeled examples per class: **8 conditions, 600 predictions**.

Create `artifacts/jev-numeric-expansion-colab.zip` locally with `python scripts/export_expanded_numeric_colab.py`, then upload that exact source/data bundle below. The notebook pins the bundle, helper, presets and frozen core hashes before executing uploaded code. Source weights are public and cached with HF authentication explicitly disabled. No API keys, hosted calls, LoRA training or Drive mounting are used. Colab GPU quota and model download time still apply.

The helper keeps the frozen numeric-label-plus-EOS likelihood scorer, uses a fixed system message with thinking disabled, and refuses context overflow or device/precision fallback. All prompts and candidate lengths are checked before each model's first prediction. Model families run sequentially.

[Protocol](../docs/EXPANDED_NUMERIC_PROTOCOL.md) · [Reproduction notes](../docs/EXPANDED_NUMERIC_REPRODUCTION.md)


In [ ]:
# Upload and verify the source/data bundle before extracting or executing it.
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib, io, json, os, stat, tempfile, zipfile

EXPECTED_BUNDLE_SHA = "f09e8181b5c6238cdcb13a842e57857b161446ff800377ab782d88fe405bd405"
EXPECTED_HELPER_SHA = "b0c07e82e03b79040f13bd7810baf92bcd3a30fe8aa3b48d0704baedea8801a3"
EXPECTED_PRESETS_SHA = "ed176a77f40fd8cd842672193333559a8221b3ac9dd52259d275c54da4cf6421"
EXPECTED_CORE_SHA = "d5547ccde4224e182653315d81c0a631c789bbe294ad7bcf95ef0cddd25ce608"
PREFIX = "jev-numeric-expansion"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

def require(condition, message):
    if not condition:
        raise ValueError(message)

def sha(content):
    return hashlib.sha256(content).hexdigest()

uploaded = files.upload()
require(len(uploaded) == 1, "Upload exactly one source ZIP")
bundle = next(iter(uploaded.values()))
require(sha(bundle) == EXPECTED_BUNDLE_SHA, "Source ZIP differs from the pinned bundle; stop and inspect")
with zipfile.ZipFile(io.BytesIO(bundle)) as archive:
    inventory = archive.infolist()
    names = [item.filename for item in inventory]
    require(len(names) == len(set(names)), "Duplicate archive member")
    require(sum(item.file_size for item in inventory) <= 20_000_000, "Unexpected source archive size")
    for item in inventory:
        name = PurePosixPath(item.filename)
        require(not name.is_absolute() and ".." not in name.parts and "\\" not in item.filename,
                "Unsafe source archive path")
        require(name.parts[0] == PREFIX and not item.is_dir() and not stat.S_ISLNK(item.external_attr >> 16),
                "Unexpected source archive member")
    manifest = json.loads(archive.read(f"{PREFIX}/bundle_manifest.json"))
    expected_names = {f"{PREFIX}/{name}" for name in manifest["files_sha256"]}
    require(set(names) == expected_names | {f"{PREFIX}/bundle_manifest.json"}, "Bundle inventory differs")
    require(manifest["core_sha256"] == EXPECTED_CORE_SHA, "Frozen core pin differs")
    require(manifest["files_sha256"]["scripts/run_expanded_numeric_local.py"] == EXPECTED_HELPER_SHA,
            "Helper source pin differs")
    require(manifest["files_sha256"]["configs/expanded_numeric_models.json"] == EXPECTED_PRESETS_SHA,
            "Model preset pin differs")
    verified = {}
    for name, expected in manifest["files_sha256"].items():
        content = archive.read(f"{PREFIX}/{name}")
        require(sha(content) == expected, f"Source member hash differs: {name}")
        verified[name] = content
    scratch = Path(tempfile.mkdtemp(prefix="jev-expansion-", dir="/content"))
    PROJECT = scratch / PREFIX
    PROJECT.mkdir()
    for name, content in verified.items():
        destination = PROJECT / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(content)
    (PROJECT / "bundle_manifest.json").write_text(json.dumps(manifest, indent=2))
del uploaded, bundle, verified
os.chdir(PROJECT)
print("Verified project:", PROJECT)
print("Source members:", len(manifest["files_sha256"]), "| Frozen core:", EXPECTED_CORE_SHA)


In [ ]:
# Install the pinned inference stack without changing the source or presets.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev,neural]",
                "transformers==4.57.6", "huggingface-hub==0.36.2"], cwd=PROJECT, check=True)

# Some Colab images bundle an incompatible optional torchao build. This
# float16 experiment does not use torchao or quantization. Repair only that case.
smoke_code = ("from transformers.models.llama.modeling_llama import LlamaForCausalLM; "
              "from transformers.models.granite.modeling_granite import GraniteForCausalLM")
smoke = subprocess.run([sys.executable, "-c", smoke_code], cwd=PROJECT, capture_output=True, text=True)
torchao_removed = False
if smoke.returncode and "torchao" in smoke.stderr:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
    torchao_removed = True
    smoke = subprocess.run([sys.executable, "-c", smoke_code], cwd=PROJECT, capture_output=True, text=True)
if smoke.returncode:
    print(smoke.stderr)
    smoke.check_returncode()
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_expanded_numeric_local.py"],
               cwd=PROJECT, check=True)


In [ ]:
# Confirm CUDA and record the actual runtime before running all eight conditions.
import importlib.metadata, platform
import torch
require(torch.cuda.is_available(), "CUDA is unavailable; select a GPU runtime. No CPU fallback is allowed.")
require(importlib.metadata.version("transformers") == "4.57.6", "Transformers version differs")
require(importlib.metadata.version("huggingface-hub") == "0.36.2", "Hub version differs")
COMMAND = [sys.executable, "scripts/run_expanded_numeric_local.py", "--device", "cuda",
           "--model-keys", "smollm2", "granite", "--shots", "0", "4",
           "--datasets", "breast_cancer", "wine", "--allow-download", "--execute"]
packages = {}
for package in ("torch", "transformers", "huggingface-hub", "accelerate", "peft", "numpy", "scipy", "scikit-learn"):
    try:
        packages[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        pass
runtime = {"schema_version": 1, "source_bundle_sha256": EXPECTED_BUNDLE_SHA,
           "helper_sha256": EXPECTED_HELPER_SHA, "preset_sha256": EXPECTED_PRESETS_SHA,
           "core_sha256": EXPECTED_CORE_SHA, "python": platform.python_version(),
           "packages": packages, "device": "cuda", "dtype": "float16",
           "gpu_name": torch.cuda.get_device_name(0),
           "gpu_total_memory_bytes": torch.cuda.get_device_properties(0).total_memory,
           "torchao_removed_after_import_failure": torchao_removed,
           "command": COMMAND[1:], "expected_conditions": 8, "expected_predictions": 600,
           "hosted_calls": 0, "adapter_training": False}
local = PROJECT / "results/numeric_expansion/local"
(local / "plans").mkdir(parents=True, exist_ok=True)
(local / "plans/colab-runtime-preflight.json").write_text(json.dumps(runtime, indent=2) + "\n")
print(json.dumps(runtime, indent=2))
subprocess.run(COMMAND, cwd=PROJECT, check=True)


In [ ]:
# Audit complete runs, export only local result artifacts, and download the ZIP.
sys.path[:0] = [str(PROJECT / "src"), str(PROJECT / "scripts")]
from run_expanded_numeric_local import audit_source_run
from tabular_data import load_native_prepared

expected = {(dataset, model, shots) for dataset in ("breast_cancer", "wine")
            for model in ("smollm2", "granite") for shots in (0, 4)}
seen = set()
export_paths = []
for run_path in sorted(local.glob("*/run.json")):
    record = json.loads(run_path.read_text())
    condition = (record["dataset"], record["config"]["renderer"]["model_key"],
                 record["config"]["shots_per_class"])
    require(condition in expected and condition not in seen, "Unexpected or duplicate run condition")
    dataset, _ = load_native_prepared(PROJECT / "data/tabular-full" / condition[0])
    audit_source_run(run_path, dataset, condition[1], condition[2])
    require({path.name for path in run_path.parent.iterdir()} ==
            {"run.json", "predictions.jsonl", "test_manifest.json"}, "Unexpected files in a result run")
    seen.add(condition)
    export_paths.extend(run_path.with_name(name) for name in ("run.json", "predictions.jsonl", "test_manifest.json"))
require(seen == expected, "Not all eight audited conditions are complete; do not publish partial results as final")
export_paths.extend(sorted((local / "plans").glob("*-preflight.json")))
require(all(path.is_file() and not path.is_symlink() and path.resolve().is_relative_to(local.resolve())
            for path in export_paths), "Unsafe result member")
archive_path = PROJECT.parent / "jev-numeric-expansion-results.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(export_paths):
        archive.write(path, arcname=path.relative_to(PROJECT).as_posix())
print("Audited conditions:", len(seen), "| Result files:", len(export_paths))
print("Archive SHA256:", sha(archive_path.read_bytes()))
files.download(str(archive_path))


After downloading, import from the repository on your local computer:

```bash
.venv/bin/python scripts/import_expanded_numeric_colab.py /path/to/jev-numeric-expansion-results.zip --execute
```

The importer independently checks archive paths and every run, then refuses to replace differing evidence. Its import record preserves the ZIP and member SHA256 hashes. The notebook exports local predictions, run/test manifests and runtime/preflight metadata only. It does not export model weights, credentials, hosted results or uploaded source data.

A fresh runtime creates a fresh experiment directory. If a run is interrupted while the same runtime remains available, rerun the inference cell to resume the frozen helper's saved row checkpoint; do not rerun the upload cell if you intend to keep that directory. An out-of-memory failure stops execution; change to a sufficiently large CUDA runtime rather than changing precision or truncating inputs. Cross-hardware numerical results need not be bit-identical; actual package/device metadata is retained.
